In [ ]:
"""
George Sedgwick 2026. Gaussian HMM and RandomForest Classifier Comparison for Regime Detection on Equities Markets.

Can unsupervised or supervised machine learning techniques provide improved risk-adjusted returns when accounting for realistic
commissions and slippage?

1. Gaussian Hidden Markov Regime Detector:

Testing momentum strategies sizing positions with the following RiskManagement system:
            - Set available cash limit (10% per position * 5 open positions = 50% total exposure)
            - Floor division of latest open price (T+1 of order time)
            - Position size multiplier based on the weighted probability of each of the following regimes:

            1 * P(BULL) + 0 * P(BEAR) + 0.2 * P(TRANSITION) + 0.7 * P(RECOVERY)

But how are these probabilities derived, and how are these regimes defined?

"""

In [ ]:
"""
DERIVING THE REGIME PROBABILITIES:

Using hmmlearn's GaussianHMM, I had originally developed a model for my event-driven backtester to use Baum-Welch
to train the model on past SPY price data and past VIX data. Then called .predict() to use Viterbi to decode the best
path and therefore the most probable regime label.

I eventually moved away from fixed regime labelling and switched to the .predict_proba() function which uses the 
Forward-Backward algorithm to look across all possible paths and provides the posterior probabilities of each latent
state.

This allowed me to use the weighted probability to size positions:
            
            1 * P(BULL) + 0 * P(BEAR) + 0.2 * P(TRANSITION) + 0.7 * P(RECOVERY

This is discussed further in the IMPLEMENTATION section of the notebook.

"""


In [ ]:
"""
DEFINING THE REGIMES:

The hidden states that the model finds are then used labelled based on means of the features in each state.


BULL - High Mean Returns + Low Mean Realised Vol
BEAR - Negative/Lowest Mean Returns
RECOVERY - High Returns + High Vol
TRANSITION - Remaining regime (model uncertain)

These are all taken from the output of the Gaussian Model with function .means_()




Notes:
After running the RFC, I am able to see the importance of features. The most significant feature of the RFC in accurate
regime identification is volatility. With hindsight, it is possible that both of the models are identifying
volatility clusters as opposed to true hidden regimes.

"""

In [ ]:
"""
STRATEGY:

- Loop through S&P 500 constituents and rank on past 126 bar momentum (1/2 the trading days in a year)
- Long the top 5
- Reshuffle every 10 days



Notes:
I tested this strategy using 12 month's returns less the latest month's, which had some academic backing at reducing noise in signals,
however after extensive testing I concluded no confirmation to this hypothesis.

126 days is used, in combination with shorter rebalancing frequencies, I had found that a 12 month momentum was too slow to create
meaningful signals while quarterly rebalances were too fast and were caught in whipsaws. Commissions are modelled in all the tests
alongside a size-weigthed price impact slippage model, therefore all results reflect appropriate penalisation for the higher frequency of trading.

Long-only is used as a result of persistent confirmation of short signals reducing risk-adjusted returns.

"""



In [ ]:
"""

GAUSSIAN MODEL IN PRACTICE: IMPLEMENTATION, ASSUMPTIONS, & LIMITATIONS

IMPLEMENTATION
Implementation involved developing an hmmlearn Gaussian model into my event-driven backtester.


BACKTESTER ENGINE LOOP:
    ---> MARKET DATA ARRIVES
    |
    ---> QUEUED ORDERS EXECUTED
    |
    ---> *REGIME DETECTOR UPDATES* <--- *NEW*
    |
    ---> STRATEGY CREATES SIGNALS
    |
    ---> PORTFOLIO CREATES ORDERS
    |
    ---> ORDERS ARE QUEUED FOR NEXT DAY OPEN EXECUTION

So the regime detector object is called with the function .update() and returns a 
tuple containing the label of the highest probable current regime, the probability
of this regime, and the position size multiplier shown above.

I wrote .update() to function slightly differently in 2 scenarios, depending on whether
a retraining of the model is due. Originally, the regime detector used Viterbi through hmmlearn's
.predict() method to produce the most likely path, and therfore the most probable regime, however
I wanted to shift from a definite state model to an dynamic exposure model, which has been achieved
through the regime based position sizing.

First the model is trained using the Baum-Welch Algorithm. Under the hood this is
required for the Markov Model as we do not know the initial probabilities,
transition probabilities, or emission probabilities. The Forward-Backward Algorithm
is therefore a requirememt as the model uses random probabilities to predict what the hidden
states were, and goes back and forth until it converges to a stable solution (or doesn't).

This happens only when retraining of the model is due (fixed interval set at instantiation
of the regime detector object). In the second scenario, when the model is not due for retrain
the early version of the model used the Viterbi algorithm, which stores the highest probable
path up until yesterday's observation, and provided the most probable regime for today given yesterday.

Instead, the model now uses .predict_proba() which returns the transition probabilities (hidden state probabilities)
and this is then used in the weighted formula to provide a position sizing multiplier.



    






LIMITATIONS AND ASSUMPTIONS
Why Gaussian?

A gaussian model was used as despite knowing financial returns are well-documented as exhibiting fat tails and
deviations from normality, the model balances computational load with information output, using a full covariance
matrix for each latent state, allowing for the labelling based on these properties.

Additionally, the general and further literature surrounding regime detection in markets using markovs supports the
use of a gaussian emission distribution as a well-reasearched and comparable benchmark for latent-state identification. Deeper
research indicates the potential for enhanced risk-adjusted returns from a gaussian-mix model, leaving a space
for extending this research in the future.






"""










In [ ]:
"""
RESULTS AND IDENTIFICATIONS









"""

In [44]:
"""
RFC
"""
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import os
import pandas as pd
import numpy as np
from datetime import date

csv_dir = "~/python-projects/ed-backtest/backtester/data/sp_constituents/SPY.csv"
filepath = os.path.join(csv_dir)
df = pd.read_csv(filepath)
csv_dir = "~/python-projects/ed-backtest/backtester/data/sp_constituents/^VIX.csv"
path = os.path.join(csv_dir)
iv_df = pd.read_csv(path)


df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.date
iv_df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.date





df['Returns'] = df['Close'].pct_change()
df['Vol'] = df['Returns'].rolling(window=20).std()
df['20_Day_Momentum'] = df['Close'].pct_change(20)
df['60_Day_Momentum'] = df['Close'].pct_change(60)
df['30_Day_Mean_Returns'] = df['Returns'].rolling(window=30).mean()
df['Skew'] = df['Returns'].rolling(60).skew()
df['Drawdown'] = df['Close'] / df['Close'].cummax() - 1
df = df.merge(iv_df[['Date', 'Close']].rename(columns={'Close': 'VIX_Close'}), on='Date', how='left')
df['30_Day_Mean_IV'] = df['VIX_Close'].rolling(window=30).mean()

df['30_Day_Mean_IV'].describe()

df['Future_Return'] = df['Close'].shift(-20) / df['Close'] - 1
df['Future_Vol'] = df['Returns'].rolling(20).std().shift(-20)


conditions = [
    (df['Future_Return'] > 0.05) & (df['Future_Vol'] < df['Future_Vol'].median()),
    (df['Future_Return'] < -0.05),
    (df['Future_Return'] > 0) & (df['Future_Vol'] > df['Future_Vol'].median())
]

choices = [
    'BULL',
    'BEAR',
    'RECOVERY'
]

df['Regime'] = np.select(
    conditions,
    choices,
    default="TRANSITION"
)



df.replace([np.inf, -np.inf], np.nan, inplace=True)

df.dropna(inplace=True)

X = df[['Returns', 'Vol', '20_Day_Momentum', '60_Day_Momentum','30_Day_Mean_Returns', 'Skew', 'Drawdown', '30_Day_Mean_IV']]
y = df['Regime']

csv_length = len(df)


split = int(csv_length * 0.75)


X_train, X_test = X[:split], X[split:]



y_train, y_test = y[:split], y[split:]




model = RandomForestClassifier(n_estimators=500, max_depth=5, random_state=0, class_weight='balanced')

model.fit(X_train, y_train)

print(f'Correct predicition (%): {accuracy_score(y_test, model.predict(X_test), normalize=True) * 100.0}')

report = classification_report(y_test, model.predict(X_test))

print(report)
print(model.feature_importances_)






Correct predicition (%): 49.8502994011976
              precision    recall  f1-score   support

        BEAR       0.09      0.23      0.13       118
        BULL       0.08      0.12      0.09        95
    RECOVERY       0.51      0.36      0.42       356
  TRANSITION       0.76      0.65      0.70       767

    accuracy                           0.50      1336
   macro avg       0.36      0.34      0.34      1336
weighted avg       0.59      0.50      0.53      1336

[0.02447736 0.19689702 0.05460979 0.11051487 0.07747322 0.12528549
 0.19607683 0.21466542]
